candidates_data = {
    'Name': ['John Doe', 'Jane Smith', 'Bob Johnson'],
    'Email': ['john@example.com', 'jane@example.com', 'bob@example.com'],
    'Mobile number': ['123-456-7890', '234-567-8901', '345-678-9012'],
    'Skills': [
        'Python, Django, React, PostgreSQL, AWS, Docker',
        'JavaScript, React, Node.js, MongoDB, AWS, Agile',
        'Python, Flask, Vue.js, MySQL, Azure, Kubernetes'
    ],
    'Total experience in years': [7, 5, 4],
    'Work done in previous company': [
        'Led a team of 5 developers to build a payment processing system',
        'Developed a customer-facing e-commerce application with React',
        'Created RESTful APIs for a financial data aggregation service'
    ],
    'College name': ['MIT', 'Stanford', 'UC Berkeley'],
    'Degree': ['MS Computer Science', 'BS Computer Science', 'BS Software Engineering'],
    'Designation': ['Senior Developer', 'Full Stack Developer', 'Backend Developer'],
    'Company names': [
        'Fintech Solutions, Tech Innovators',
        'E-commerce Giants, Web Development Co',
        'Software Solutions, Data Systems Inc'
    ]
}

In [24]:
candidates_data = {
    'Name': ['John Doe', 'Jane Smith', 'Bob Johnson'],
    'Email': ['john@example.com', 'jane@example.com', 'bob@example.com'],
    'Mobile number': ['123-456-7890', '234-567-8901', '345-678-9012'],
    'Skills': [
        'Python, Django, React, PostgreSQL, AWS, Docker',
        'JavaScript, React, Node.js, MongoDB, AWS, Agile',
        'Python, Flask, Vue.js, MySQL, Azure, Kubernetes'
    ],
    'Total experience in years': [7, 5, 4],
    'Work done in previous company': [
        'Led a team of 5 developers to build a payment processing system',
        'Developed a customer-facing e-commerce application with React',
        'Created RESTful APIs for a financial data aggregation service'
    ],
    'College name': ['MIT', 'Stanford', 'UC Berkeley'],
    'Degree': ['MS Computer Science', 'BS Computer Science', 'BS Software Engineering'],
    'Designation': ['Senior Developer', 'Full Stack Developer', 'Backend Developer'],
    'Company names': [
        'Fintech Solutions, Tech Innovators',
        'E-commerce Giants, Web Development Co',
        'Software Solutions, Data Systems Inc'
    ]
}

In [25]:
import warnings
warnings.filterwarnings('ignore')
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import yaml
from dotenv import load_dotenv
import json
from textwrap import dedent
from crewai import Agent, Crew, Task , Process
from datetime import datetime
from pydantic import BaseModel, Field
from typing import List, Dict, Optional


class FinalRecommendation(BaseModel):
    technical_score: float
    industry_score: float
    experience_score: float
    education_score: float
    key_strengths: List[str] = Field(description="List of candidate's key strengths relevant to the role")
    areas_of_concern: List[str] = Field(description="List of potential concerns or gaps in the candidate's profile")
    match_category: str = Field(description="One of 'Strong Match' (>75), 'Potential Match' (50-75), 'Not a Match' (<50)")
    relative_ranking: Optional[str] = Field(None, description="Optional ranking relative to other candidates")
    justification: str = Field(description="Detailed justification for the recommendation")
    recommendation: str = Field(description="Clear hiring recommendation")

class ResumeScreeningSystem:
    def __init__(self, job_description, llm):
        """
        Initialize the Resume Screening System with a job description.
        
        Args:
            job_description (str): The job description to match candidates against
            llm: The language model to use for agents
        """
        self.job_description = job_description
        self.llm = llm
        self.setup_agents()
        self.setup_tasks()
        self.setup_crew()
        
    def setup_agents(self):
        """Set up the specialized agents for resume screening"""
        
        # Technical Skills Assessor
        self.technical_skills_agent = Agent(
            role="Technical Skills Assessor",
            goal="Evaluate candidate's technical skills against job requirements",
            backstory=dedent("""
                You are an expert in evaluating technical skills in resumes.
                Your expertise is in mapping technical skills mentioned in resumes
                to those required in job descriptions. You understand various
                technologies, programming languages, tools, and frameworks.
                You evaluate each candidate independently based on their merits.
            """),
            verbose=True,
            allow_delegation=False,
            llm=self.llm
        )
        
        # Work Experience Evaluator
        self.work_experience_agent = Agent(
            role="Work Experience Evaluator",
            goal="Assess the relevance and quality of candidate's work experience",
            backstory=dedent("""
                You analyze work experience to determine if it aligns with the job.
                You look for relevant projects, responsibilities, and achievements
                that would translate well to the position. You can detect patterns
                of career progression and determine if the candidate has the right
                industry experience. You evaluate each candidate independently.
            """),
            verbose=True,
            allow_delegation=False,
            llm=self.llm
        )
        
        # Education Qualification Assessor
        self.education_agent = Agent(
            role="Education Qualification Assessor",
            goal="Evaluate the candidate's educational background",
            backstory=dedent("""
                You assess educational qualifications to determine if they meet
                the job requirements. You understand different degree types,
                educational institutions, and how they relate to industry standards.
                You can evaluate the relevance of the candidate's academic background.
                You evaluate each candidate independently based on their qualifications.
            """),
            verbose=True,
            allow_delegation=False,
            llm=self.llm
        )
        
        # Industry Fit Evaluator
        self.industry_agent = Agent(
            role="Industry Fit Evaluator",
            goal="Assess if the candidate has relevant industry experience",
            backstory=dedent("""
                You specialize in evaluating industry-specific experience and knowledge.
                You understand different industries, their specific requirements,
                terminology, and best practices. You can determine if a candidate's
                background aligns with the target industry. You evaluate each candidate
                independently based on their own industry experience.
            """),
            verbose=True,
            allow_delegation=False,
            llm=self.llm
        )
        
        # Senior Resume Analyst
        self.senior_analyst = Agent(
            role="Senior Resume Analyst",
            goal="Provide a comprehensive evaluation and final recommendation",
            backstory=dedent("""
                As a senior hiring analyst, you review all aspects of a candidate's
                profile and make a final recommendation. You can synthesize information 
                from multiple specialized assessments and provide a holistic evaluation. 
                Your expertise lies in understanding how different aspects of a profile 
                contribute to job fit. You evaluate each candidate purely on their own 
                merits without comparison to others unless specifically asked to rank them.
            """),
            verbose=True,
            allow_delegation=False,
            llm=self.llm
        )
    
    def setup_tasks(self):
        """Set up the tasks for each agent"""
        
        # Technical Skills Assessment Task
        self.tech_skills_task = Task(
            description=dedent(f"""
                Analyze the technical skills listed in the candidate's profile.
                Compare them with the required skills in the job description:
                {self.job_description}
                
                Provide a detailed assessment of:
                1. Skills match percentage
                2. Critical skills present
                3. Critical skills missing
                4. Bonus skills the candidate has
                
                IMPORTANT: Score the candidate on technical skills from 0-100.
                
                Output your analysis in JSON format with the following structure:
                {{
                    "skills_match_percentage": number,
                    "critical_skills_present": [list of skills],
                    "critical_skills_missing": [list of skills],
                    "bonus_skills": [list of skills],
                    "technical_score": number (0-100),
                    "detailed_assessment": string
                }}
            """),
            agent=self.technical_skills_agent,
            expected_output="A evaluation of  Technical experience in structured JSON format"

        )
        
        # Work Experience Assessment Task
        self.work_exp_task = Task(
            description=dedent(f"""
                Evaluate the candidate's work experience against the job requirements:
                {self.job_description}
                
                Analyze:
                1. Years of relevant experience
                2. Relevance of previous roles and responsibilities
                3. Achievements and projects that align with the job
                4. Career progression and growth
                
                IMPORTANT: Score the candidate on work experience from 0-100.
                
                Output your analysis in JSON format with the following structure:
                {{
                    "years_relevant_experience": number,
                    "relevance_of_roles": string,
                    "key_achievements": [list of achievements],
                    "career_progression": string,
                    "experience_score": number (0-100),
                    "detailed_assessment": string
                }}
            """),
            agent=self.work_experience_agent,
            expected_output="A evaluation of  work experience experience in structured JSON format"

        )
        
        # Education Assessment Task
        self.edu_task = Task(
            description=dedent(f"""
                Assess the candidate's educational qualifications against the job requirements:
                {self.job_description}
                
                Analyze:
                1. Degree relevance
                2. Institution reputation
                3. Specializations and majors
                4. Additional certifications and training
                
                IMPORTANT: Score the candidate on education from 0-100.
                
                Output your analysis in JSON format with the following structure:
                {{
                    "degree_relevance": string,
                    "institution_quality": string,
                    "relevant_specializations": [list of specializations],
                    "additional_certifications": [list of certifications],
                    "education_score": number (0-100),
                    "detailed_assessment": string
                }}
            """),
            agent=self.education_agent,
            expected_output="A evaluation of educational background in structured JSON format"

        )
        
        # Industry Fit Assessment Task
        self.industry_task = Task(
            description=dedent(f"""
                Evaluate the candidate's industry fit based on their profile:
                {self.job_description}
                
                Analyze:
                1. Industry-specific experience
                2. Knowledge of industry trends and practices
                3. Experience with industry-specific tools and methodologies
                4. Alignment with industry culture and values
                
                IMPORTANT: Score the candidate on industry fit from 0-100.
                
                Output your analysis in JSON format with the following structure:
                {{
                    "industry_experience": string,
                    "industry_knowledge": string,
                    "industry_tools_experience": [list of tools],
                    "culture_alignment": string,
                    "industry_score": number (0-100),
                    "detailed_assessment": string
                }}
            """),
            agent=self.industry_agent,
            expected_output="A evaluation of industry relevant experience in structured JSON format"
        )
        
        # Final Analysis Task
        self.final_analysis_task = Task(
            description=dedent(f"""
                Review all the assessments provided by the specialized agents.
                Synthesize the information to provide a comprehensive evaluation of the candidate.
                
                You will receive: 
                - Technical Skills Assessment (score out of 100)
                - Work Experience Assessment (score out of 100)
                - Education Assessment (score out of 100)
                - Industry Fit Assessment (score out of 100)
                
                IMPORTANT: Using these individual assessments, determine:
                1. The candidate's key strengths (extract from high-scoring areas)
                2. Areas of concern (extract from low-scoring areas)
                3. Match category based on overall impression:
                   - "Strong Match" (avg score >75)
                   - "Potential Match" (avg score 50-75)
                   - "Not a Match" (avg score <50)
                4. A detailed justification for your recommendation
                5. A clear hiring recommendation
                
                DO NOT include overall score or weighted scores in your analysis.
                
                Provide your analysis in the following format:
                {{  "technical_score": number (0-100),
                    "industry_score": number (0-100),
                    "experience_score": number (0-100),
                    "education_score": number (0-100),
                    "key_strengths": [list of strengths],
                    "areas_of_concern": [list of concerns],
                    "match_category": string (category),
                    "justification": string,
                    "recommendation": string
                }}
                
                Be specific in your assessment and make a clear recommendation based on
                the job description: {self.job_description}
            """),
            agent=self.senior_analyst,
            expected_output="A final comprehensive evaluation in structured JSON format",
            context=[
                self.tech_skills_task,
                self.work_exp_task,
                self.edu_task,
                self.industry_task
            ],
            output_pydantic=FinalRecommendation
        )
        

    def setup_crew(self):
        """Set up the crew with the defined agents and tasks"""
        self.crew = Crew(
            agents=[
                self.technical_skills_agent,
                self.work_experience_agent,
                self.education_agent,
                self.industry_agent,
                self.senior_analyst
            ],
            tasks=[
                self.tech_skills_task,
                self.work_exp_task,
                self.edu_task,
                self.industry_task,
                self.final_analysis_task
            ],
            process=Process.sequential
        )
    
    def evaluate_candidate(self, candidate_data):
        """
        Evaluate a single candidate against the job description
        
        Args:
            candidate_data (dict): Dictionary containing candidate information
            
        Returns:
            dict: Evaluation results
        """
        # Prepare candidate info for agents
        candidate_info = f"""
        Candidate Profile:
        Name: {candidate_data['Name']}
        Email: {candidate_data['Email']}
        Mobile: {candidate_data['Mobile number']}
        Skills: {candidate_data['Skills']}
        Total Experience: {candidate_data['Total experience in years']} years
        Previous Work: {candidate_data['Work done in previous company']}
        College: {candidate_data['College name']}
        Degree: {candidate_data['Degree']}
        Designation: {candidate_data['Designation']}
        Companies: {candidate_data['Company names']}
        """
        
        # Update task descriptions with candidate info
        self.tech_skills_task.description += f"\n\nCandidate Information:\n{candidate_info}"
        self.work_exp_task.description += f"\n\nCandidate Information:\n{candidate_info}"
        self.edu_task.description += f"\n\nCandidate Information:\n{candidate_info}"
        self.industry_task.description += f"\n\nCandidate Information:\n{candidate_info}"
        
        # Run the crew
        result = self.crew.kickoff()
        
        
        return result

Evaluating candidate 1/3: John Doe


In [15]:
job_description = """
We are looking for a Senior Full Stack Developer with 5+ years of experience.
Required skills:
- Python, Django, Flask
- JavaScript, React, Redux
- SQL, PostgreSQL
- AWS, Docker, Kubernetes
- CI/CD pipelines

The ideal candidate has experience in fintech or e-commerce industry, has a
Bachelor's degree in Computer Science or related field, and has led small
development teams.
"""
screener = ResumeScreeningSystem(job_description,llm)
result = screener.evaluate_candidate(candidate_data_input)

2025-04-07 00:11:52,116 - 15320 - __init__.py-__init__:518 - WARNING: Overriding of current TracerProvider is not allowed




> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 
{
    "skills_match_percentage": 80,
    "critical_skills_present": [
        "Python", 
        "Django", 
        "React", 
        "PostgreSQL", 
        "AWS", 
        "Docker"
    ],
    "critical_skills_missing": [
        "Flask", 
        "JavaScript", 
        "Redux", 
        "Kubernetes", 
        "CI/CD pipelines"
    ],
    "bonus_skills": [
        "Leadership experience", 
        "Payment processing system development", 
        "Fintech industry experience"
    ],
    "technical_score": 85,
    "detailed_assessment": "The candidate, John Doe, has a strong technical background with 7 years of experience in the fintech industry. He has experience with critical skills such as Python, Django, React, PostgreSQL, AWS, and Docker. However, he is missing some critical skills like Flask, JavaScript, Redux, Kubernetes, and CI/CD pipelines. He has bonus skills like leadership experience, p

In [1]:

from langchain_openai import ChatOpenAI

In [ ]:
import os

from crewai import LLM
import pandas as pd


job_description = """
We are looking for a Senior Full Stack Developer with 5+ years of experience.
Required skills:
- Python, Django, Flask
- JavaScript, React, Redux
- SQL, PostgreSQL
- AWS, Docker, Kubernetes
- CI/CD pipelines

The ideal candidate has experience in fintech or e-commerce industry, has a
Bachelor's degree in Computer Science or related field, and has led small
development teams.
"""

llm = LLM(model=os.getenv("CREWAI_LLM_MODEL", "azure/gpt-4o-mini"))
screener = ResumeScreeningSystem(job_description,llm)

email = []
Technical_Score = []
Experience_Score = []
Education_Score = []
Industry_Score = []
Match_Category = []
Key_Strengths = []
Areas_of_Concern = []
Recommendation = []

# Loop through each candidate (currently using .head(1), you can change it to iterate through all)
for idx, row in candidates_df.iterrows():
    print(f"Evaluating candidate {idx+1}/{len(candidates_df)}: {row['Name']}")
    candidate_data_input = row.to_dict()
    result = screener.evaluate_candidate(candidate_data_input)

    # Append each field to its respective list
    email.append(row['Email'])
    Technical_Score.append(result.technical_score)
    Experience_Score.append(result.experience_score)
    Education_Score.append(result.education_score)
    Industry_Score.append(result.industry_score)
    Match_Category.append(result.match_category)
    Key_Strengths.append(result.key_strengths)
    Areas_of_Concern.append(result.areas_of_concern)
    Recommendation.append(result.recommendation)

# Create a DataFrame with all collected results
dff_result = pd.DataFrame({
    'email': email,
    'Technical_Score': Technical_Score,
    'Experience_Score': Experience_Score,
    'Education_Score': Education_Score,
    'Industry_Score': Industry_Score,
    'Match_Category': Match_Category,
    'Key_Strengths': Key_Strengths,
    'Areas_of_Concern': Areas_of_Concern,
    'Recommendation': Recommendation
})
final_result_df = pd.merge(candidates_df, dff_result, left_on='Email', right_on='email', how='left')
final_result_df.drop(columns=['email'], inplace=True)






    


2025-04-07 11:18:50,441 - 15320 - __init__.py-__init__:518 - WARNING: Overriding of current TracerProvider is not allowed


Evaluating candidate 1/3: John Doe


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 
{
    "skills_match_percentage": 80,
    "critical_skills_present": [
        "Python", 
        "Django", 
        "React", 
        "PostgreSQL", 
        "AWS", 
        "Docker"
    ],
    "critical_skills_missing": [
        "Flask", 
        "JavaScript", 
        "Redux", 
        "Kubernetes", 
        "CI/CD pipelines"
    ],
    "bonus_skills": [],
    "technical_score": 85,
    "detailed_assessment": "The candidate, John Doe, has a strong technical background with 7 years of experience in the fintech industry. He has experience with Python, Django, React, PostgreSQL, AWS, and Docker, which are critical skills required for the Senior Full Stack Developer position. However, he is missing skills in Flask, JavaScript, Redux, Kubernetes, and CI/CD pipelines. His experience in leading a team of 5 developers to build a payment processing system and his MS deg

In [27]:
final_result_df 

,Name,Email,Mobile number,Skills,Total experience in years,Work done in previous company,College name,Degree,Designation,Company names,Technical_Score,Experience_Score,Education_Score,Industry_Score,Match_Category,Key_Strengths,Areas_of_Concern,Recommendation
0,John Doe,john@example.com,123-456-7890,"Python, Django, React, PostgreSQL, AWS, Docker",7,Led a team of 5 developers to build a payment ...,MIT,MS Computer Science,Senior Developer,"Fintech Solutions, Tech Innovators",85.0,90.0,95.0,92.0,Strong Match,[Strong technical background with 7 years of e...,"[Missing skills in Flask, JavaScript, Redux, K...","Hire the candidate, John Doe, as a Senior Full..."
1,Jane Smith,jane@example.com,234-567-8901,"JavaScript, React, Node.js, MongoDB, AWS, Agile",5,Developed a customer-facing e-commerce applica...,Stanford,BS Computer Science,Full Stack Developer,"E-commerce Giants, Web Development Co",80.0,90.0,95.0,90.0,Strong Match,"[Strong background in Python and Django, Exper...","[Missing experience in JavaScript, Flask, SQL,...",I strongly recommend hiring John Doe for the S...
2,Bob Johnson,bob@example.com,345-678-9012,"Python, Flask, Vue.js, MySQL, Azure, Kubernetes",4,Created RESTful APIs for a financial data aggr...,UC Berkeley,BS Software Engineering,Backend Developer,"Software Solutions, Data Systems Inc",85.0,90.0,95.0,92.0,Strong Match,[Strong technical background with experience i...,"[Lack of experience in JavaScript, Flask, Redu...",I strongly recommend John Doe for the Senior F...
